In [ ]:
import numpy as np
import os
import pickle
from pathlib import Path
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# CIFAR-10 loader
def _unpickle(file):
    with open(file, 'rb') as fo:
        return pickle.load(fo, encoding='bytes')

def load_cifar10_raw(cifar_dir):
    p = Path(cifar_dir)
    train_imgs, train_lbls = [], []
    for i in range(1, 6):
        batch = _unpickle(p / f'data_batch_{i}')
        train_imgs.append(batch[b'data'])
        train_lbls.extend(batch[b'labels'])
    train_imgs = np.concatenate(train_imgs, axis=0)

    test_batch = _unpickle(p / 'test_batch')
    test_imgs = test_batch[b'data']
    test_lbls = test_batch[b'labels']

    train_imgs = train_imgs.reshape(-1, 3, 32, 32)
    test_imgs  = test_imgs.reshape(-1, 3, 32, 32)

    return (train_imgs, np.array(train_lbls)), (test_imgs, np.array(test_lbls))

def make_cifar10_loaders(cifar_dir='./cifar-10-batches-py', batch_size=64):
    (X_tr, y_tr), (X_te, y_te) = load_cifar10_raw(cifar_dir)

    X_tr = X_tr.astype(np.float32) / 255.0
    X_te = X_te.astype(np.float32) / 255.0
    X_tr = (X_tr - 0.5) / 0.5
    X_te = (X_te - 0.5) / 0.5

    train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr).long())
    test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te).long())

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader  = DataLoader(test_ds, batch_size=1000, shuffle=False)
    return train_loader, test_loader

# alternative downloading of data if not in temp-storage at Colab
if not Path('./cifar-10-batches-py').exists():
    !wget -q http://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz
    !tar -xzf cifar-10-python.tar.gz

train_loader_cifar, val_loader_cifar = make_cifar10_loaders('./cifar-10-batches-py')
print(f"CIFAR-10 train batches: {len(train_loader_cifar)}")

# settings
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

EPOCHS   = 20
PLOT_DIR = './plots_ex2'
os.makedirs(PLOT_DIR, exist_ok=True)

activation = nn.ReLU()

# model
class CIFAR_CNN(nn.Module):
    def __init__(self, activation):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(128*4*4, 512)
        self.fc2   = nn.Linear(512, 10)
        self.act   = activation
    def forward(self, x):
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = x.view(-1, 128*4*4)
        x = self.act(self.fc1(x))
        return self.fc2(x)

# training loop
def train_model(model, train_loader, val_loader, optimizer,
                dataset_name="CIFAR10", exp_name="exp"):
    criterion = nn.CrossEntropyLoss()

    t_loss, v_loss, t_acc, v_acc = [], [], [], []

    for epoch in range(EPOCHS):
        # train
        model.train()
        tr_l, tr_c = 0.0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            tr_l += loss.item()
            tr_c += (out.argmax(1) == y).sum().item()
        t_loss.append(tr_l / len(train_loader))
        t_acc.append(tr_c / len(train_loader.dataset))

        # validate
        model.eval()
        val_l, val_c = 0.0, 0
        with torch.no_grad():
            for X, y in val_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                val_l += criterion(out, y).item()
                val_c += (out.argmax(1) == y).sum().item()
        v_loss.append(val_l / len(val_loader))
        v_acc.append(val_c / len(val_loader.dataset))

        print(f"{dataset_name} [{exp_name}] Epoch {epoch+1:02d}/{EPOCHS} | "
              f"TrainLoss {t_loss[-1]:.4f} ValLoss {v_loss[-1]:.4f} | "
              f"TrainAcc {t_acc[-1]:.4f} ValAcc {v_acc[-1]:.4f}")

    return t_loss, v_loss, t_acc, v_acc

# plotting
def plot_and_save(exp_name, tl, vl, ta, va):
    epochs = range(1, len(tl)+1)
    plt.figure(figsize=(12,5))

    plt.subplot(1,2,1)
    plt.plot(epochs, tl, label='Train loss', marker='o')
    plt.plot(epochs, vl, label='Val loss',   marker='s')
    plt.title(f'{exp_name} – loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid()

    plt.subplot(1,2,2)
    plt.plot(epochs, ta, label='Train acc', marker='o')
    plt.plot(epochs, va, label='Val acc',   marker='s')
    plt.title(f'{exp_name} – accuracy')
    plt.xlabel('Epoch'); plt.ylabel('Acc'); plt.legend(); plt.grid()

    plt.tight_layout()
    fname = f"{PLOT_DIR}/{exp_name}_curves.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved → {fname}\n")

# SGD tuning
print("\nSGD TUNING ON CIFAR-10 \n")

lrs       = [0.001, 0.01, 0.1]
momentums = [0.0, 0.5, 0.9]

best_val_acc = 0.0
best_params  = None
best_curves  = None

for lr in lrs:
    for mom in momentums:
        exp_name = f"SGD_lr{lr}_mom{mom}"
        print(f"\n--- {exp_name} ---")
        model = CIFAR_CNN(activation)
        model.to(device)
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=mom)
        tl, vl, ta, va = train_model(model, train_loader_cifar,
                                     val_loader_cifar, optimizer,
                                     exp_name=exp_name)

        final_val = va[-1]
        print(f"Final Val Acc: {final_val:.4f}")

        if final_val > best_val_acc:
            best_val_acc = final_val
            best_params  = (lr, mom)
            best_curves  = (tl, vl, ta, va)

# plotting the best SGD
if best_curves:
    plot_and_save(f"Best_SGD_lr{best_params[0]}_mom{best_params[1]}", *best_curves)

print(f"\nBest SGD → lr={best_params[0]}, momentum={best_params[1]}, ValAcc={best_val_acc:.4f}")

# comparing optimizers
print("\n=== PART 2: COMPARE OPTIMIZERS ===\n")

optimizers_info = {
    'SGD'    : {'func': optim.SGD,      'has_mom': True},
    'RMSProp': {'func': optim.RMSprop,  'has_mom': True},
    'AdaGrad': {'func': optim.Adagrad,  'has_mom': False},
    'Adam'   : {'func': optim.Adam,     'has_mom': False}
}

results = {}
curves   = {}

for name, info in optimizers_info.items():
    if name == 'SGD':
        results[name] = best_val_acc
        curves[name]  = best_curves
        continue

    print(f"\n--- {name} ---")
    best_acc = 0.0
    best_cfg = None
    best_c   = None

    for lr in lrs:
        mom_vals = momentums if info['has_mom'] else [0]
        for mom in mom_vals:
            exp_name = f"{name}_lr{lr}_mom{mom}" if info['has_mom'] else f"{name}_lr{lr}"

            model = CIFAR_CNN(activation)
            model.to(device)

            # optimizer creation
            if name == 'AdaGrad':
                optimizer = info['func'](model.parameters(), lr=lr, eps=1e-8)
            elif name == 'RMSProp':
                optimizer = info['func'](model.parameters(), lr=lr, momentum=mom)
            elif name == 'Adam':
                optimizer = info['func'](model.parameters(), lr=lr)
            else:  # SGD
                optimizer = info['func'](model.parameters(), lr=lr, momentum=mom)

            tl, vl, ta, va = train_model(model, train_loader_cifar,
                                         val_loader_cifar, optimizer,
                                         exp_name=exp_name)

            final = va[-1]
            if final > best_acc:
                best_acc = final
                best_cfg = (lr, mom) if info['has_mom'] else lr
                best_c   = (tl, vl, ta, va)

    results[name] = best_acc
    curves[name]  = best_c
    plot_and_save(f"Best_{name}", *best_c)
    print(f"Best {name} → ValAcc {best_acc:.4f} (config {best_cfg})")

# final summary
# -------------------------------------------------
print("\n" + "="*60)
print("FINAL COMPARISON")
print("="*60)
for n, a in results.items():
    print(f"{n:8}: {a:.4f}")

CIFAR-10 train batches: 782
Using device: cuda

SGD TUNING ON CIFAR-10 


--- SGD_lr0.001_mom0.0 ---
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 01/20 | TrainLoss 2.3016 ValLoss 2.2997 | TrainAcc 0.1014 ValAcc 0.1220
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 02/20 | TrainLoss 2.2976 ValLoss 2.2953 | TrainAcc 0.1533 ValAcc 0.1717
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 03/20 | TrainLoss 2.2927 ValLoss 2.2896 | TrainAcc 0.1849 ValAcc 0.1863
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 04/20 | TrainLoss 2.2857 ValLoss 2.2808 | TrainAcc 0.1867 ValAcc 0.1853
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 05/20 | TrainLoss 2.2740 ValLoss 2.2651 | TrainAcc 0.1946 ValAcc 0.2018
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 06/20 | TrainLoss 2.2524 ValLoss 2.2350 | TrainAcc 0.2084 ValAcc 0.2195
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 07/20 | TrainLoss 2.2106 ValLoss 2.1786 | TrainAcc 0.2213 ValAcc 0.2306
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 08/20 | TrainLoss 2.1468 ValLoss 2.1125 | TrainAcc 0.2360 ValAcc 0.2494
CIFAR10 [SGD_lr0.001_mom0.0] Epoch 09/20 | 